# Benchmark: Registration Methods (Barrier Post-Processing)

Same registration methods as the SLSQP benchmark, but post-processes
folding with the penalty -> log-barrier L-BFGS-B solver.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
import time
import numpy as np
import matplotlib.pyplot as plt

from dvfopt import jacobian_det2D
from dvfopt.core.iterative2d_barrier import iterative_2d_barrier
from dvfopt.viz import plot_grid_before_after
from benchmark_utils import run_correction


## Generate test images

In [ ]:
def make_test_images(H=64, W=64):
    yy, xx = np.mgrid[:H, :W].astype(np.float32)
    cy, cx = H / 2, W / 2
    r = np.sqrt((yy - cy)**2 + (xx - cx)**2)
    moving = ((r < H * 0.35).astype(np.float32)
              + (r < H * 0.15).astype(np.float32))
    fixed = np.roll(moving, (3, 5), axis=(0, 1))
    return moving, fixed

moving, fixed = make_test_images()
fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
axes[0].imshow(moving, cmap="gray"); axes[0].set_title("Moving")
axes[1].imshow(fixed, cmap="gray"); axes[1].set_title("Fixed")
plt.tight_layout(); plt.show()


## Run registration methods + barrier correction

In [ ]:
methods = {}

# --- Demons (SimpleITK) ---
try:
    import SimpleITK as sitk
    m_sitk = sitk.GetImageFromArray(moving[None])
    f_sitk = sitk.GetImageFromArray(fixed[None])
    demons = sitk.DiffeomorphicDemonsRegistrationFilter()
    demons.SetNumberOfIterations(200)
    demons.SetStandardDeviations(1.5)
    t0 = time.time()
    disp_sitk = demons.Execute(f_sitk, m_sitk)
    t_reg = time.time() - t0
    disp_np = sitk.GetArrayFromImage(disp_sitk)
    dvf = np.zeros((3, 1, moving.shape[0], moving.shape[1]), dtype=np.float64)
    dvf[2, 0] = disp_np[0, :, :, 0]
    dvf[1, 0] = disp_np[0, :, :, 1]
    methods["Demons"] = {"dvf": dvf, "t_reg": t_reg}
    print(f"Demons: {t_reg:.2f}s")
except ImportError:
    print("SimpleITK not available, skipping Demons")

# --- B-spline (SimpleITK) ---
try:
    import SimpleITK as sitk
    m_sitk = sitk.GetImageFromArray(moving[None])
    f_sitk = sitk.GetImageFromArray(fixed[None])
    tx = sitk.BSplineTransformInitializer(f_sitk, [4, 4, 1])
    reg = sitk.ImageRegistrationMethod()
    reg.SetInitialTransform(tx, True)
    reg.SetMetricAsMeanSquares()
    reg.SetOptimizerAsLBFGSB()
    reg.SetOptimizerScalesFromPhysicalShift()
    t0 = time.time()
    final_tx = reg.Execute(f_sitk, m_sitk)
    t_reg = time.time() - t0
    disp_filter = sitk.TransformToDisplacementFieldFilter()
    disp_filter.SetReferenceImage(f_sitk)
    disp_sitk = disp_filter.Execute(final_tx)
    disp_np = sitk.GetArrayFromImage(disp_sitk)
    dvf = np.zeros((3, 1, moving.shape[0], moving.shape[1]), dtype=np.float64)
    dvf[2, 0] = disp_np[0, :, :, 0]
    dvf[1, 0] = disp_np[0, :, :, 1]
    methods["B-spline"] = {"dvf": dvf, "t_reg": t_reg}
    print(f"B-spline: {t_reg:.2f}s")
except (ImportError, RuntimeError) as e:
    print(f"B-spline skipped: {e}")

print(f"\nAvailable methods: {list(methods.keys())}")


In [ ]:
results = {}
for name, m in methods.items():
    dvf = m["dvf"]
    phi_init = np.stack([dvf[-2, 0], dvf[-1, 0]])
    jac = jacobian_det2D(phi_init)
    n_neg = int((jac <= 0).sum())
    if n_neg == 0:
        print(f"{name}: no negatives, skipping")
        continue
    r = run_correction(dvf, iterative_2d_barrier)
    r["t_reg"] = m["t_reg"]
    results[name] = r
    status = "OK" if r["n_neg_final"] == 0 else "FAIL"
    print(f"{name}: neg {r['n_neg_init']}->{r['n_neg_final']}  "
          f"min {r['min_jdet']:+.6f}  L2={r['l2_err']:.4f}  "
          f"t_reg={m['t_reg']:.2f}s  t_corr={r['time']:.2f}s  [{status}]")


## Results

In [ ]:
if results:
    names = list(results.keys())
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
    x = np.arange(len(names)); w = 0.35
    ax = axes[0]
    ax.bar(x-w/2, [results[n]["n_neg_init"] for n in names], w, label="Before", color="tab:red", alpha=0.7)
    ax.bar(x+w/2, [results[n]["n_neg_final"] for n in names], w, label="After", color="tab:blue", alpha=0.7)
    ax.set_xticks(x); ax.set_xticklabels(names); ax.set_ylabel("Neg Jdet"); ax.set_title("Neg Jdet Before vs After"); ax.legend()
    ax = axes[1]
    ax.bar(x-w/2, [results[n]["t_reg"] for n in names], w, label="Registration", color="tab:green")
    ax.bar(x+w/2, [results[n]["time"] for n in names], w, label="Barrier correction", color="tab:orange")
    ax.set_xticks(x); ax.set_xticklabels(names); ax.set_ylabel("Time (s)"); ax.set_title("Registration vs Correction Time"); ax.legend()
    ax = axes[2]
    ax.bar(names, [results[n]["l2_err"] for n in names], color="tab:purple")
    ax.set_ylabel("L2 deviation"); ax.set_title("L2 Deviation")
    plt.suptitle("Registration + Barrier Post-Processing", fontsize=13)
    plt.tight_layout(); plt.show()
else:
    print("No methods produced folding")


In [ ]:
for name, r in results.items():
    deformation_i = np.zeros((3, 1) + r["phi_init"].shape[-2:])
    deformation_i[1, 0] = r["phi_init"][0]
    deformation_i[2, 0] = r["phi_init"][1]
    plot_grid_before_after(deformation_i, r["phi"], figsize=(14, 6))
    plt.suptitle(f"{name} - Barrier Correction", fontsize=12)
    plt.show()
